In [1]:
import sys
sys.path.append('..')
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report,f1_score

from xgboost import XGBClassifier

from src.text_preprocess import clean_text_ml

[nltk_data] Downloading package wordnet to /Users/rishabh/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/rishabh/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rishabh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
import mlflow
import mlflow.sklearn

In [12]:
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

from src.config_mlflow import setup_mlflow

setup_mlflow()

MLflow connected successfully


In [13]:
mlflow.set_experiment("ML_Model_Type_Classification")

2026/06/10 14:19:53 INFO mlflow.tracking.fluent: Experiment with name 'ML_Model_Type_Classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/rishabh/Desktop/Customer Support Intelligence/mlruns/4', creation_time=1781081393930, experiment_id='4', last_update_time=1781081393930, lifecycle_stage='active', name='ML_Model_Type_Classification', tags={}, trace_location=None, workspace='default'>

In [2]:
df=pd.read_csv('../data/ticket_classification.csv')
df=df.dropna()

In [3]:
df['text'] = df['subject'].apply(clean_text_ml)+" "+df['body'].apply(clean_text_ml)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['body'].iloc[0])
print("\nAFTER:\n", df['text'].iloc[0])

BEFORE:
 Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

AFTER:
 account disruption dear customer support team , writing report significant problem centralized account management portal , currently appears offline . outage blocking access account setting , leading substantial inconvenience . attempted log multiple time using different browser device , issue persists . could please provide update outage status estimated time resolution ? also , alternative way access manage account downtime ?


In [4]:
le_type = LabelEncoder()

df['label'] = le_type.fit_transform(df['type'])


In [10]:
joblib.dump(
    le_type,
    "../models/type_label_encoder.pkl"
)

['../models/type_label_encoder.pkl']

In [5]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df['text'])

y = df['label']

In [9]:
joblib.dump(
    tfidf,
    "../models/tfidf_type.pkl"
)

['../models/tfidf_type.pkl']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
model_name = "XGBoost_Classifier"

with mlflow.start_run(run_name=model_name):
    model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    max_depth=10,
    n_jobs=-1,
    n_estimators=100
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)

    mlflow.log_param("max_depth", 10)
    mlflow.log_param("vectorizer", "TF-IDF")
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("eval_metric", "mlogloss")
    
    # Log all metrics to MLflow
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    
    # Save model artifact
    mlflow.sklearn.log_model(model, artifact_path=model_name)
    print("Accuracy:",
                accuracy_score(y_test, predictions))

    print("F1 Score:",
        f1_score(y_test, predictions, average='macro'))

    print(classification_report(y_test, predictions))
        


2026/06/10 14:28:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 14:28:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.8785540211210398
F1 Score: 0.8727328706531985
              precision    recall  f1-score   support

           0       0.98      0.90      0.94       511
           1       0.82      0.93      0.87      1957
           2       0.82      0.62      0.70      1028
           3       0.97      0.99      0.98      1428

    accuracy                           0.88      4924
   macro avg       0.90      0.86      0.87      4924
weighted avg       0.88      0.88      0.87      4924

🏃 View run XGBoost_Classifier at: http://127.0.0.1:5000/#/experiments/4/runs/603a1202a68347d6b7ee03869f9aff9b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [7]:
model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    max_depth=10,
    n_jobs=-1,
    n_estimators=100
    )
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
print("Accuracy:",
                accuracy_score(y_test, predictions))

print("F1 Score:",
        f1_score(y_test, predictions, average='macro'))

print(classification_report(y_test, predictions))

Accuracy: 0.8785540211210398
F1 Score: 0.8727328706531985
              precision    recall  f1-score   support

           0       0.98      0.90      0.94       511
           1       0.82      0.93      0.87      1957
           2       0.82      0.62      0.70      1028
           3       0.97      0.99      0.98      1428

    accuracy                           0.88      4924
   macro avg       0.90      0.86      0.87      4924
weighted avg       0.88      0.88      0.87      4924



In [8]:
joblib.dump(
    model,
    "../models/type_xgb.pkl"
)

['../models/type_xgb.pkl']